<a href="https://colab.research.google.com/github/arushikumarics24-beep/BIS_LAB/blob/main/grey_wolf_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np

# Generator data
a = np.array([0.004, 0.006, 0.009])
b = np.array([5.3, 5.5, 5.8])
c = np.array([500, 400, 200])

# Generator limits (MW)
Pmin = np.array([50, 50, 50])
Pmax = np.array([200, 150, 100])

# Total power demand (MW)
demand = 350

# Objective function: Total generation cost
def cost_function(P):
    return np.sum(a * P**2 + b * P + c)


# Repair function to satisfy power demand
def repair(P):
    P = np.clip(P, Pmin, Pmax)

    diff = demand - np.sum(P)

    if diff > 0:
        capacity = Pmax - P
        if np.sum(capacity) > 0:
            P += diff * capacity / np.sum(capacity)

    elif diff < 0:
        capacity = P - Pmin
        if np.sum(capacity) > 0:
            P += diff * capacity / np.sum(capacity)

    return np.clip(P, Pmin, Pmax)


# Grey Wolf Optimizer
def GWO(n_wolves, max_iter, dim):

    # Initialize wolf population
    wolves = np.random.uniform(Pmin, Pmax, (n_wolves, dim))

    # Repair initial population
    for i in range(n_wolves):
        wolves[i] = repair(wolves[i])

    # Main GWO loop
    for iteration in range(max_iter):

        # Calculate fitness
        fitness = np.array([
            cost_function(wolves[i])
            for i in range(n_wolves)
        ])

        # Sort wolves according to fitness
        sorted_indices = np.argsort(fitness)

        # Identify Alpha, Beta and Delta
        alpha = wolves[sorted_indices[0]].copy()
        beta = wolves[sorted_indices[1]].copy()
        delta = wolves[sorted_indices[2]].copy()

        # Decrease parameter a from 2 to 0
        A_factor = 2 - 2 * iteration / max_iter

        # Update wolf positions
        new_wolves = np.zeros((n_wolves, dim))

        for i in range(n_wolves):

            for j in range(dim):

                # Alpha wolf
                r1, r2 = np.random.random(), np.random.random()
                A1 = 2 * A_factor * r1 - A_factor
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha[j] - wolves[i, j])
                X1 = alpha[j] - A1 * D_alpha

                # Beta wolf
                r1, r2 = np.random.random(), np.random.random()
                A2 = 2 * A_factor * r1 - A_factor
                C2 = 2 * r2
                D_beta = abs(C2 * beta[j] - wolves[i, j])
                X2 = beta[j] - A2 * D_beta

                # Delta wolf
                r1, r2 = np.random.random(), np.random.random()
                A3 = 2 * A_factor * r1 - A_factor
                C3 = 2 * r2
                D_delta = abs(C3 * delta[j] - wolves[i, j])
                X3 = delta[j] - A3 * D_delta

                # Update position
                new_wolves[i, j] = (X1 + X2 + X3) / 3

            # Apply generator limits and power balance
            new_wolves[i] = repair(new_wolves[i])

        wolves = new_wolves

        print("Iteration:", iteration + 1,
              "Best Cost:", round(np.min([
                  cost_function(w) for w in wolves
              ]), 4))

    # Find final best solution
    fitness = np.array([
        cost_function(w) for w in wolves
    ])

    best_index = np.argmin(fitness)

    best_generation = wolves[best_index]
    best_cost = fitness[best_index]

    return best_generation, best_cost


# Main program
n_wolves = 20
max_iter = 100
dim = 3

best_generation, best_cost = GWO(
    n_wolves, max_iter, dim
)

# Display results
print("\n========== OPTIMAL POWER GENERATION ==========")

for i in range(dim):
    print("Generator", i + 1, ":",
          round(best_generation[i], 2), "MW")

print("\nTotal Power Generated:",
      round(np.sum(best_generation), 2), "MW")

print("Power Demand:", demand, "MW")

print("Minimum Generation Cost:",
      round(best_cost, 2))


Iteration: 1 Best Cost: 3241.5139
Iteration: 2 Best Cost: 3241.4063
Iteration: 3 Best Cost: 3241.7514
Iteration: 4 Best Cost: 3241.3048
Iteration: 5 Best Cost: 3241.2272
Iteration: 6 Best Cost: 3241.3737
Iteration: 7 Best Cost: 3241.5984
Iteration: 8 Best Cost: 3241.4268
Iteration: 9 Best Cost: 3241.7521
Iteration: 10 Best Cost: 3242.2995
Iteration: 11 Best Cost: 3241.8637
Iteration: 12 Best Cost: 3241.32
Iteration: 13 Best Cost: 3241.5104
Iteration: 14 Best Cost: 3241.5571
Iteration: 15 Best Cost: 3241.949
Iteration: 16 Best Cost: 3241.7566
Iteration: 17 Best Cost: 3241.2614
Iteration: 18 Best Cost: 3241.185
Iteration: 19 Best Cost: 3242.2229
Iteration: 20 Best Cost: 3241.3113
Iteration: 21 Best Cost: 3241.218
Iteration: 22 Best Cost: 3241.479
Iteration: 23 Best Cost: 3241.3649
Iteration: 24 Best Cost: 3241.4436
Iteration: 25 Best Cost: 3241.4226
Iteration: 26 Best Cost: 3241.5657
Iteration: 27 Best Cost: 3241.2541
Iteration: 28 Best Cost: 3242.4739
Iteration: 29 Best Cost: 3241.4651
